In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import pickle
import os

df_model = pd.read_csv('processed/df_model.csv')
df_orig = pd.read_csv('processed/classification_with_orf.tsv', sep='\t')

with open('processed/sequences.pkl', 'rb') as f:
    seqs = pickle.load(f)

df_full = df_orig[['isoform']].copy()
df_full['sequence'] = df_full['isoform'].map(seqs)
df_full = pd.concat([df_full, df_model], axis=1)

df_full['original_class'] = df_full['target_4class']
df_full = df_full.drop(columns=['target_4class', 'target_3class'])

df_full = df_full.dropna(subset=['sequence'])

df_full = df_full[df_full['sequence'].str.len() <= 8000].reset_index(drop=True)

for col in df_full.columns:
    if col in ['isoform', 'sequence', 'original_class']:
        continue
    if df_full[col].dtype == object:
        df_full[col] = df_full[col].fillna(df_full[col].mode()[0])
    else:
        df_full[col] = df_full[col].fillna(df_full[col].median())

print(len(df_full))
print(df_full['original_class'].value_counts())

479737
original_class
real_high    199562
uncertain    171119
artifact      64804
real_mid      44252
Name: count, dtype: int64


In [3]:
def encode_sequence(seq, max_len):
    mapping = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
    encoded = [mapping[c] if c in mapping else -1 for c in seq[:max_len]]
    padded = encoded + [-1] * (max_len - len(encoded))
    return torch.tensor(padded, dtype=torch.long)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2, kernel_size=7):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(2)
        )

    def forward(self, x):
        return self.block(x)

class SeqEncoder(nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.convs = nn.Sequential(
            ConvBlock(4, 128, kernel_size=7, dropout=dropout),
            ConvBlock(128, 256, kernel_size=5, dropout=dropout),
            ConvBlock(256, hidden_dim, kernel_size=3, dropout=dropout),
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        mask = (x == -1)
        one_hot = torch.nn.functional.one_hot(x.clamp(min=0), num_classes=4).float()
        one_hot[mask] = 0
        x = one_hot.permute(0, 2, 1)
        x = self.convs(x)
        x = self.global_pool(x)
        return x.squeeze(-1)

class TabEncoder(nn.Module):
    def __init__(self, input, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.mlp(x)

class IsoformClassifier(nn.Module):
    def __init__(self, tab_input, tab_dim, conv_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.tab = TabEncoder(input=tab_input, hidden_dim=tab_dim, dropout=dropout)
        self.seq = SeqEncoder(conv_dim, dropout=dropout)
        self.mlp = nn.Sequential(
            nn.Linear(conv_dim + tab_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_conv, x_tab):
        x_conv = self.seq(x_conv)
        x_tab = self.tab(x_tab)
        comb = torch.cat([x_conv, x_tab], dim=1)
        return self.mlp(comb).squeeze(-1)


In [5]:
device = 'cuda'
model = IsoformClassifier(tab_input=21, tab_dim=64, conv_dim=128, hidden_dim=128)
model.load_state_dict(torch.load('best_model.pt'))
model = model.to('cuda')
model.eval()

IsoformClassifier(
  (tab): TabEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=21, out_features=64, bias=True)
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=64, out_features=64, bias=True)
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.3, inplace=False)
    )
  )
  (seq): SeqEncoder(
    (convs): Sequential(
      (0): ConvBlock(
        (block): Sequential(
          (0): Conv1d(4, 128, kernel_size=(7,), stride=(1,), padding=(3,))
          (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
          (3): Dropout(p=0.3, inplace=False)
          (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        )
      )
      (1): ConvBlock(
        (block): Sequential(
         

In [6]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, f1_score
from tqdm import tqdm
import numpy as np


TAB_FEATURES = [
    'strand', 'length', 'exons', 'ref_length', 'ref_exons',
    'diff_to_gene_TSS', 'diff_to_gene_TTS', 'RTS_stage', 'all_canonical',
    'bite', 'perc_A_downstream_TTS', 'protein_length', 'has_orf',
    'FSM_class_A', 'FSM_class_B', 'FSM_class_C',
    'orf_type_3prime_partial', 'orf_type_5prime_partial', 'orf_type_complete',
    'orf_type_internal', 'orf_type_no_orf'
]

In [7]:
class IsoformInferenceDataset(torch.utils.data.Dataset):
    def __init__(self, df, max_len=8000):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = encode_sequence(row['sequence'], self.max_len)
        tab = torch.tensor(row[TAB_FEATURES].values.astype(np.float32))
        return seq, tab

model.load_state_dict(torch.load('best_model.pt'))
model = model.to(device)
model.eval()

loader = DataLoader(IsoformInferenceDataset(df_full), batch_size=64, shuffle=False, num_workers=4)

all_scores = []
with torch.no_grad():
    for seq, tab in tqdm(loader, desc='Inference'):
        seq, tab = seq.to(device), tab.to(device)
        logits = model(seq, tab)
        scores = torch.sigmoid(logits).cpu().numpy()
        all_scores.extend(scores)

df_full['score'] = all_scores

print(df_full.groupby('original_class')['score'].describe())

Inference: 100%|██████████| 7496/7496 [05:53<00:00, 21.20it/s]


                   count      mean       std           min       25%  \
original_class                                                         
artifact         64804.0  0.130264  0.209665  5.238597e-11  0.000405   
real_high       199562.0  0.935631  0.165218  5.830372e-05  0.980467   
real_mid         44252.0  0.502616  0.307021  1.914811e-06  0.234467   
uncertain       171119.0  0.349098  0.289878  3.170066e-13  0.088431   

                     50%       75%       max  
original_class                                
artifact        0.004056  0.199539  1.000000  
real_high       0.994625  0.999389  0.999980  
real_mid        0.521129  0.782625  0.999596  
uncertain       0.292892  0.551566  0.999911  


In [ ]:
train_df = pd.read_parquet('processed/train.parquet')
val_df = pd.read_parquet('processed/val.parquet')
test_df = pd.read_parquet('processed/test.parquet')

print(f"train: {len(train_df):,}")
print(f"val:   {len(val_df):,}")
print(f"test:  {len(test_df):,}")

train_ids = set(train_df['isoform'])
val_ids = set(val_df['isoform'])
test_ids = set(test_df['isoform'])

print(f"train ∩ val:  {len(train_ids & val_ids)}")
print(f"train ∩ test: {len(train_ids & test_ids)}")
print(f"val ∩ test:   {len(val_ids & test_ids)}")

train_seqs = set(train_df['sequence'])
val_seqs = set(val_df['sequence'])
test_seqs = set(test_df['sequence'])

print(f"train ∩ val:  {len(train_seqs & val_seqs)}")
print(f"train ∩ test: {len(train_seqs & test_seqs)}")
print(f"val ∩ test:   {len(val_seqs & test_seqs)}")

print(f"всего: {len(train_df)}, уникальных: {train_df['sequence'].nunique()}")

numeric_cols = train_df.select_dtypes(include=[np.number]).columns
correlations = train_df[numeric_cols].corr()['label'].abs().sort_values(ascending=False)
print(correlations)

train: 188,528
val:   53,892
test:  26,906

=== Пересечения по isoform ID ===
train ∩ val:  0
train ∩ test: 0
val ∩ test:   0

=== Пересечения по sequences ===
train ∩ val:  157
train ∩ test: 69
val ∩ test:   21

=== Дубликаты sequence в train ===
всего: 188528, уникальных: 188262

=== Корреляция признаков с лейблом ===
label                      1.000000
all_canonical              0.608530
FSM_class_C                0.443929
FSM_class_A                0.359837
length                     0.347756
FSM_class_B                0.261003
ref_exons                  0.197195
orf_type_3prime_partial    0.128797
diff_to_gene_TSS           0.119104
orf_type_internal          0.100258
has_orf                    0.077535
orf_type_no_orf            0.077535
orf_type_5prime_partial    0.073766
diff_to_gene_TTS           0.068059
orf_type_complete          0.061073
protein_length             0.047195
exons                      0.046569
ref_length                 0.038179
perc_A_downstream_TTS      0.0

In [10]:
class TabOnlyClassifier(nn.Module):
    def __init__(self, tab_input, tab_dim=64, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.tab = TabEncoder(input=tab_input, hidden_dim=tab_dim, dropout=dropout)
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_tab):
        x = self.tab(x_tab)
        return self.mlp(x).squeeze(-1)


class TabOnlyDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tab = torch.tensor(row[TAB_FEATURES].values.astype(np.float32))
        label = torch.tensor(row['label'], dtype=torch.float32)
        return tab, label


def train_epoch_tab(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_labels, all_preds = [], []
    
    pbar = tqdm(loader, desc='Train', leave=False)
    for tab, label in pbar:
        tab, label = tab.to(device), label.to(device)
        optimizer.zero_grad()
        logits = model(tab)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        all_labels.extend(label.cpu().numpy())
        all_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    auc = roc_auc_score(all_labels, all_preds)
    return total_loss / len(loader), auc


@torch.no_grad()
def eval_epoch_tab(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_labels, all_preds = [], []
    
    for tab, label in loader:
        tab, label = tab.to(device), label.to(device)
        logits = model(tab)
        loss = criterion(logits, label)
        total_loss += loss.item()
        all_labels.extend(label.cpu().numpy())
        all_preds.extend(torch.sigmoid(logits).cpu().numpy())
    
    auc = roc_auc_score(all_labels, all_preds)
    f1 = f1_score(all_labels, [p > 0.5 for p in all_preds])
    return total_loss / len(loader), auc, f1


train_loader_tab = DataLoader(TabOnlyDataset(train_df), batch_size=256, shuffle=True, num_workers=4)
val_loader_tab   = DataLoader(TabOnlyDataset(val_df),   batch_size=256, shuffle=False, num_workers=4)

model_tab = TabOnlyClassifier(tab_input=21).to(device)
pos_weight = torch.tensor([66905 / 202421]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = Adam(model_tab.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(5):
    train_loss, train_auc = train_epoch_tab(model_tab, train_loader_tab, optimizer, criterion, device)
    val_loss, val_auc, val_f1 = eval_epoch_tab(model_tab, val_loader_tab, criterion, device)
    print(f"Epoch {epoch+1} | train auc: {train_auc:.4f} | val auc: {val_auc:.4f} f1: {val_f1:.4f}")

Epoch 1 | train auc: 0.9676 | val auc: 0.9841 f1: 0.9466


Epoch 2 | train auc: 0.9803 | val auc: 0.9841 f1: 0.9474


Epoch 3 | train auc: 0.9828 | val auc: 0.9882 f1: 0.9532


Epoch 4 | train auc: 0.9841 | val auc: 0.9891 f1: 0.9550


Epoch 5 | train auc: 0.9850 | val auc: 0.9893 f1: 0.9503


In [11]:
model_tab.eval()

class TabOnlyInferenceDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tab = torch.tensor(row[TAB_FEATURES].values.astype(np.float32))
        return tab

loader_full_tab = DataLoader(TabOnlyInferenceDataset(df_full), batch_size=256, shuffle=False, num_workers=4)

all_scores_tab = []
with torch.no_grad():
    for tab in tqdm(loader_full_tab, desc='Tab-only inference'):
        tab = tab.to(device)
        logits = model_tab(tab)
        scores = torch.sigmoid(logits).cpu().numpy()
        all_scores_tab.extend(scores)

df_full['score_tab_only'] = all_scores_tab

print("=== Tab-only MLP model ===")
print(df_full.groupby('original_class')['score_tab_only'].describe())

Tab-only inference: 100%|██████████| 1874/1874 [00:34<00:00, 54.62it/s]

=== Tab-only MLP model ===
                   count      mean       std           min       25%  \
original_class                                                         
artifact         64804.0  0.097984  0.153986  3.892209e-14  0.000180   
real_high       199562.0  0.922861  0.202841  9.812711e-05  0.987844   
real_mid         44252.0  0.497902  0.322587  1.387651e-04  0.217091   
uncertain       171119.0  0.251025  0.221025  1.691102e-13  0.069853   

                     50%       75%       max  
original_class                                
artifact        0.001150  0.164579  0.997154  
real_high       0.996688  0.999424  0.999981  
real_mid        0.435169  0.858978  0.998526  
uncertain       0.199500  0.375679  0.999490  
